In [1]:
from trainModel import trainModel
from Dataset import ModelDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix, accuracy_score
import pandas as pd
import os
from generateSplits import generateSplits
from torch.utils.data import DataLoader
import torch,gc
import numpy as np
from typing import Literal, Callable, Iterator

In [2]:
DATASET_PATH = r"E:\SRP\SRP-2025-Project\ISPY2_T0_T3_DCE_npz"

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
dataset_df = pd.read_excel(r"e:\SRP\ISPY2-Data-Collector\ISPY2-Imaging-Cohort-1-Clinical-Data.xlsx")
dataset_df = dataset_df.set_index("Patient_ID",drop=True)
dataset_df = dataset_df.loc[dataset_df.index.isin([int(os.path.splitext(fname)[0].replace("ISPY2-","")) for fname in os.listdir(DATASET_PATH)]),["HR","HER2","pCR"]]

train_df, test_df = generateSplits(dataset_df,0.2,seed=2008)
skf = StratifiedKFold(n_splits=4,shuffle=True,random_state=2008)

In [ ]:
best_params = {'lr': 0.010203264127576701, 'weight_decay': 0.004617603477095672, 'batch_size': 8, 'optimiser_name': 'Adam'}

In [4]:
def evaluate_roc_auc(model:torch.nn.Module,test_loader:DataLoader,combine_timepoints:bool,out_features:Literal[1,2]):
    model.eval()
    y_true=[]
    y_score=[]
    
    with torch.no_grad():
        for T0_volumes,T3_volumes,mols,labels in test_loader:
            T0_volumes = T0_volumes.to(device)
            T3_volumes = T3_volumes.to(device)
            mols = mols.to(device)
            labels = labels.to(device)
            if combine_timepoints:
                logits = model(torch.cat((T0_volumes,T3_volumes),dim=1),mols)
            else:
                logits = model(T0_volumes,T3_volumes,mols)
                
            if out_features == 1:
                score = torch.sigmoid(logits).squeeze()
            elif out_features == 2:
                score = torch.nn.functional.softmax(logits,dim=1)[:,1]
            y_true.extend(labels.cpu().numpy())
            y_score.extend(score.cpu().numpy())
    return roc_auc_score(y_true,y_score)

In [ ]:
def get_scores(model:torch.nn.Module,test_loader:DataLoader,combine_timepoints:bool,out_features:Literal[1,2]):
    model.eval()
    y_true=[]
    y_score=[]
    
    with torch.no_grad():
        for T0_volumes,T3_volumes,mols,labels in test_loader:
            T0_volumes = T0_volumes.to(device)
            T3_volumes = T3_volumes.to(device)
            mols = mols.to(device)
            labels = labels.to(device)
            if combine_timepoints:
                logits = model(torch.cat((T0_volumes,T3_volumes),dim=1),mols)
            else:
                logits = model(T0_volumes,T3_volumes,mols)
                
            if out_features == 1:
                score = torch.sigmoid(logits).squeeze()
            elif out_features == 2:
                score = torch.nn.functional.softmax(logits,dim=1)[:,1]
            y_true.extend(labels.cpu().numpy())
            y_score.extend(score.cpu().numpy())
    return y_true,y_score

In [ ]:
def score_to_pred(y_true,y_score):
    fpr,tpr,thresholds = roc_curve(y_true,y_score)
    youden_index = tpr-fpr
    best_idx = np.argmax(youden_index)
    best_threshold = thresholds[best_idx]
    y_pred = np.array(y_score)>=best_threshold
    return y_pred

In [ ]:
def four_fold_cv_train(Model_class:type[torch.nn.Module],
                       optimiser_class:Callable[[Iterator[torch.nn.Parameter]],torch.optim.Optimizer],
                       class_samples:dict,
                       num_epochs:int,
                       batch_size:int,
                       combine_timepoints:bool,
                       output_features:Literal[1,2],
                       loss_fn:Callable,
                       score_fn:Callable,
                       score_name:str,
                       probs_fn:Callable[[torch.nn.Module,DataLoader,bool,Literal[1,2]],tuple]):
    
    roc_auc_scores = []
    average_precision_scores = []
    sensitivites = []
    specificities = []
    PPVs = []
    NPVs = []
    accuracies = []
    for train_index,val_index in skf.split(train_df,train_df["pCR"]):
        model = Model_class()
        model = model.to(device)
        fold_train_df = train_df.iloc[train_index]
        fold_test_df = train_df.iloc[val_index]
        fold_train_dataset = ModelDataset(fold_train_df,DATASET_PATH,class_samples,loading_bar=False)
        fold_train_loader = DataLoader(fold_train_dataset,batch_size=batch_size,shuffle=True)
        fold_test_dataset = ModelDataset(fold_test_df,DATASET_PATH,loading_bar=False)
        fold_test_loader = DataLoader(fold_test_dataset,batch_size=batch_size)
        model,score = trainModel(model,
                                 train_loader=fold_train_loader,
                                 combine_timepoints=combine_timepoints,
                                 out_features=output_features,
                                 loss_fn = loss_fn,
                                 optimiser=optimiser_class(model.parameters()),
                                 num_epochs=num_epochs,
                                 val_loader=fold_test_loader,
                                 score_fn=score_fn,
                                 score_name=score_name,
                                 patience=num_epochs//4)
        roc_auc_scores.append(score)
        y_true,y_score = probs_fn(model,fold_test_loader,combine_timepoints,output_features)
        print(f"Average Precision={(ap_score_val:=average_precision_score(y_true,y_score)):.4f}")
        average_precision_scores.append(ap_score_val)
        y_pred = score_to_pred(y_true,y_score)
        tn,fp,fn,tp = confusion_matrix(y_true,y_pred).ravel()
        sensitivites.append((tp/(tp+fn)).item())
        specificities.append((tn/(tn+fp)).item())
        PPVs.append((tp/(tp+fp)).item())
        NPVs.append((tn/(tn+fn)).item())
        accuracies.append(accuracy_score(y_true,y_pred))
        
        del model
        
    return {"ROC_AUC":sum(roc_auc_scores)/len(roc_auc_scores),
            "Average Precision":sum(average_precision_scores)/len(average_precision_scores),
            "Sensitivity / Recall / TPR":sum(sensitivites)/len(sensitivites),
            "Specificity / TNR":sum(specificities)/len(specificities),
            "PPV / Precision":sum(PPVs)/len(PPVs),
            "NPV":sum(NPVs)/len(NPVs),
            "Accuracy":sum(accuracies)/len(accuracies)}
        

Run Above Cells

In [ ]:
from models.CMC_Model.model import Model as CMC_Model

optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.CrossEntropyLoss()
four_fold_scores = four_fold_cv_train(CMC_Model,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=True,
                                             output_features=2,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores)
print(four_fold_scores)

'''
'ROC_AUC': 0.5445866499092306, 
'Average Precision': 0.38931456533232567, 
'Sensitivity / Recall / TPR': 0.7067204301075268, 
'Specificity / TNR': 0.4179383116883117, 
'PPV / Precision': 0.4787010990875746, 
'NPV': 0.8056943056943057, 
'Accuracy': 0.5174418604651163
'''


KeyboardInterrupt: 

In [9]:
from models.CMC_SE.model import Model as CMC_SE_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.CrossEntropyLoss()
four_fold_scores = four_fold_cv_train(CMC_SE_Model,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=True,
                                             output_features=2,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores)
print(four_fold_scores)

'''
'ROC_AUC': 0.561273652422846, 
'Average Precision': 0.4331408019221398, 
'Sensitivity / Recall / TPR': 0.6610215053763441, 
'Specificity / TNR': 0.49261363636363636, 
'PPV / Precision': 0.43590604120695486, 
'NPV': 0.7934704184704184, 
'Accuracy': 0.5524281805745554
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=19.8124
ROC AUC=0.643452380952381
Epoch 1 Done. Average Loss=0.8168
ROC AUC=0.5589285714285714
Epoch 2 Done. Average Loss=0.7230
ROC AUC=0.556547619047619
Epoch 3 Done. Average Loss=0.5936
ROC AUC=0.505357142857143
Epoch 4 Done. Average Loss=0.5848
ROC AUC=0.5154761904761904
Epoch 5 Done. Average Loss=0.5707
ROC AUC=0.4622023809523809
Early stopping triggered at epoch 5. Best ROC AUC=0.643452380952381
Average Precision=0.4947
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=18.8987
ROC AUC=0.5325513196480939
Epoch 1 Done. Average Loss=0.6996
ROC AUC=0.4750733137829912
Epoch 2 Done. Average Loss=0.6752
ROC AUC=0.5120234604105572
Epoch 3 Done. Average Loss=0.6492
ROC AUC=0.5026392961876833
Epoch 4 Done. Average Loss=0.6648
ROC AUC=0.4574780058651026
Epoch 5 Done. Average Loss=0.6361
ROC AUC=0.5079178885630499
Early stopping triggered a

In [ ]:
from models.CMC_SE_with_AvgPool import Model as CMC_SE_with_AP_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.CrossEntropyLoss()
four_fold_scores = four_fold_cv_train(CMC_SE_with_AP_Model,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=True,
                                             output_features=2,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores)
print(four_fold_scores)

'''
'ROC_AUC': 0.6627363496718336, 
'Average Precision': 0.508258678020139, 
'Sensitivity / Recall / TPR': 0.660752688172043, 
'Specificity / TNR': 0.6693181818181818, 
'PPV / Precision': 0.5264654345102233, 
'NPV': 0.7861817427855164, 
'Accuracy': 0.6667920656634747
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7407
ROC AUC=0.6124999999999999
Epoch 1 Done. Average Loss=0.6905
ROC AUC=0.6898809523809524
Epoch 2 Done. Average Loss=0.6779
ROC AUC=0.6791666666666667
Epoch 3 Done. Average Loss=0.6878
ROC AUC=0.6904761904761906
Epoch 4 Done. Average Loss=0.6784
ROC AUC=0.6904761904761905
Epoch 5 Done. Average Loss=0.6672
ROC AUC=0.6791666666666667
Epoch 6 Done. Average Loss=0.6805
ROC AUC=0.6720238095238096
Epoch 7 Done. Average Loss=0.6630
ROC AUC=0.6416666666666667
Epoch 8 Done. Average Loss=0.6717
ROC AUC=0.6880952380952381
Early stopping triggered at epoch 8. Best ROC AUC=0.6904761904761906
Average Precision=0.5938
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7420
ROC AUC=0.4381231671554252
Epoch 1 Done. Average Loss=0.6970
ROC AUC=0.5859237536656892
Epoch 2 Done. Average Loss=0.6691
ROC AUC=0.4973607038123168
Epoch 3 Done. Average Lo

In [ ]:
from models.ResNet.model import Model as DualResNetModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores)
print(four_fold_scores)

'''
'ROC_AUC': 0.6992595307917888, 
'Average Precision': 0.50554374036523, 
'Sensitivity / Recall / TPR': 0.7755376344086021, 
'Specificity / TNR': 0.5966720779220779, 
'PPV / Precision': 0.5266954663693795, 
'NPV': 0.8417085295656724, 
'Accuracy': 0.6609439124487004
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7327
ROC AUC=0.6142857142857143
Epoch 1 Done. Average Loss=0.6906
ROC AUC=0.618452380952381
Epoch 2 Done. Average Loss=0.7009
ROC AUC=0.6803571428571428
Epoch 3 Done. Average Loss=0.6536
ROC AUC=0.75
Epoch 4 Done. Average Loss=0.6450
ROC AUC=0.6702380952380953
Epoch 5 Done. Average Loss=0.6404
ROC AUC=0.6720238095238096
Epoch 6 Done. Average Loss=0.6423
ROC AUC=0.5178571428571428
Epoch 7 Done. Average Loss=0.6271
ROC AUC=0.7160714285714286
Epoch 8 Done. Average Loss=0.6537
ROC AUC=0.5369047619047619
Early stopping triggered at epoch 8. Best ROC AUC=0.75
Average Precision=0.5416
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.6686
ROC AUC=0.535483870967742
Epoch 1 Done. Average Loss=0.6546
ROC AUC=0.44633431085043984
Epoch 2 Done. Average Loss=0.6255
ROC AUC=0.6070381231671554
Epoch 3 Done. Average Loss=0.6519
ROC AUC=0.441055718

In [ ]:
from models.ResNet_SE.ResNet_SE_r_16 import Model as DualResNetSE_r_16_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetSE_r_16_Model,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores)
print(four_fold_scores)

'''
'ROC_AUC': 0.6598273634967183, 
'Average Precision': 0.4810334624219552, 
'Sensitivity / Recall / TPR': 0.7448924731182796, 
'Specificity / TNR': 0.592775974025974, 
'PPV / Precision': 0.5024766899766899, 
'NPV': 0.8146424349881797, 
'Accuracy': 0.6462380300957593
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7342
ROC AUC=0.6339285714285715
Epoch 1 Done. Average Loss=0.7007
ROC AUC=0.6702380952380953
Epoch 2 Done. Average Loss=0.6674
ROC AUC=0.6601190476190476
Epoch 3 Done. Average Loss=0.6771
ROC AUC=0.6267857142857143
Epoch 4 Done. Average Loss=0.6952
ROC AUC=0.6202380952380953
Epoch 5 Done. Average Loss=0.6656
ROC AUC=0.6702380952380952
Epoch 6 Done. Average Loss=0.6885
ROC AUC=0.6238095238095238
Early stopping triggered at epoch 6. Best ROC AUC=0.6702380952380953
Average Precision=0.4752
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7160
ROC AUC=0.5824046920821114
Epoch 1 Done. Average Loss=0.6543
ROC AUC=0.509090909090909
Epoch 2 Done. Average Loss=0.6513
ROC AUC=0.5378299120234603
Epoch 3 Done. Average Loss=0.6281
ROC AUC=0.5313782991202346
Epoch 4 Done. Average Loss=0.6558
ROC AUC=0.5020527859237536
Epoch 5 Done. Average Los

In [10]:
from models.ResNet_SE.ResNet_SE_r_4 import Model as DualResNetSE_r_4_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetSE_r_4_Model,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores)
print(four_fold_scores)

ModuleNotFoundError: No module named 'models.ResNet_SE.ResNet_SE_r_4'; 'models.ResNet_SE' is not a package

In [9]:
gc.collect()

44